# Max Chelminski
## HW3: Functions

In [1]:
import pandas as pd
import numpy as np

In [2]:
flights = pd.read_csv('../../Data/nycflights.csv')

1. **Write a function that takes a single numerical variable and returns three values, the minimum number, the median, and the maximum number of the vector. Test your function using the month column of the flights data set.**

In [3]:
def min_med_max(vector):
    tmp = sorted(vector.to_list())
    minval = min(tmp)
    maxval = max(tmp)
    n = len(tmp)
    if n % 2 == 0:
        medval = (tmp[n//2] + tmp[(n//2)-1])/2
    else:
        medval = tmp[n//2]
    return minval, medval, maxval

In [4]:
minimum, median, maximum = min_med_max(flights['month'])
print(minimum)
print(median)
print(maximum)

1
7.0
12


2. **Explain your reasoning for choosing your function's name in the previous question.**

I chose the function name "min_med_max" because it concisely and clearly describes what the function returns. Perhaps med could be slightly unclear, but I think the value of brevity outweighs the value of pure clarity, as I think mistake would be rare.

3. **Write a function that categorizes a numerical variable in the flights data into four categories.**

In [5]:
def flight_time_categorizer(df, col):
    conditions =[(df[col] >= 500) & (df[col] < 1200),
        (df[col] >= 1200) & (df[col] < 1700),
        (df[col] >= 1700) & (df[col] < 2100),
        (df[col] >= 2100) | (df[col] < 500)]
    choices = ['Morning', 'Afternoon', 'Evening', 'Night']
    df['dep_time_cat'] = np.select(conditions, choices, default='Unknown')
    return df['dep_time_cat']

In [6]:
flight_time_categorizer(flights, 'dep_time')

0         Morning
1         Morning
2         Morning
3         Morning
4         Morning
           ...   
336771    Unknown
336772    Unknown
336773    Unknown
336774    Unknown
336775    Unknown
Name: dep_time_cat, Length: 336776, dtype: object

In [7]:
print(flights['dep_time_cat'].value_counts())

dep_time_cat
Morning      129539
Afternoon     98617
Evening       79793
Night         20572
Unknown        8255
Name: count, dtype: int64


4. **Explain your reasoning for choosing your function's name in the previous question.**

In slight contrast to the previous function, this function name abandons shorthand notation for maximum clarity. Because there is no common shorthand for flight time categorization, I think this name is the best balance of clarity without being too dense.

5. **Write a function that calculates the median of all numeric variables in the flights dataset.**

In [8]:
def df_medians(df):
    subset = df.select_dtypes(include=[np.number])
    medians = {}
    for column in subset.columns:
        tmp = sorted(subset[column].dropna().to_list())
        n = len(tmp)
        if n % 2 == 0:
            medval = (tmp[n//2] + tmp[(n//2)-1])/2
        else:
            medval = tmp[n//2]
        medians[column] = medval
    medians_df = pd.DataFrame([medians])
    return medians_df

In [9]:
df_medians(flights)

,Unnamed: 0,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,flight,air_time,distance,hour,minute
0,168388.5,2013.0,7.0,16.0,1401.0,1359.0,-2.0,1535.0,1556.0,-5.0,1496.0,129.0,872.0,13.0,29.0


6. **Explain your reasoning for choosing your function's name in the previous question.**

This function name is quite clear, it takes a df and outputs the medians of the numeric columns. The name is concise and clear, and was an easy choice.

7. **Modify the function t_test() we wrote in class together this week so that this function can handle violations to the homogeneity of variance (HOV) assumption. This assumption is described below in case you are not familiar with it. Please read the class activity carefully - as all of those details are relevant.**

In [10]:
import scipy.stats as stats

In [11]:
def t_test(num_var, bin_var):  
    
    # Calculate the mean difference between the two groups
    group1 = num_var[bin_var == 0]
    group2 = num_var[bin_var == 1]
    mean_diff = group1.mean() - group2.mean()
    
    # Calculate the sample sizes and sample variances of the two groups
    n1 = group1.shape[0]
    n2 = group2.shape[0]
    s2_1 = group1.var()
    s2_2 = group2.var()

    sample_var_ratio = s2_1/s2_2
    if (sample_var_ratio >= 0.25) and (sample_var_ratio <= 4.0):
        # Calculate the degrees of freedom (DF)
        DF = n1 + n2 - 2
    
        # Calculate the pooled standard deviation (sp)
        sp = (((n1-1)*s2_1+(n2-1)*s2_2)/DF)**0.5
    
        # Calculate the standard error of the mean difference
        SE_mean_diff = sp * (1/n1 + 1/n2)**0.5
    
        # Calculate the t-statistic
        t_statistic = mean_diff/SE_mean_diff
    
        # Calculate the p-value
        p_value = 2 * (1 - stats.t.cdf(np.abs(t_statistic), DF))
    
        # Put the results into a DataFrame
        results = pd.DataFrame({'Continuous Variable': [num_var.name], 'Binary Variable': [bin_var.name], 'N': [n1+n2], 'Mean Diff': [f'{mean_diff:.2f}'], 
                            'SE of Mean Diff': [f'{SE_mean_diff:.2f}'], 'DF': [DF], 't-statistic': [f'{t_statistic:.3f}'],
                            'p-value': [f'{p_value:.3f}'], 'test': ['Independent samples t-test']})
        print("The homogeneity of variance assumption was not violated, so an independent samples t-test was conducted")
        
    else:
        DF = ((s2_1/n1 + s2_2/n2)**2)/((1/(n1-1))*(s2_1/n1)**2 + (1/(n2-1))*(s2_2/n2)**2)
        
        SE_mean_diff = ((s2_1/n1) + (s2_2/n2))**0.5

        t_statistic = mean_diff/SE_mean_diff

        p_value = 2 * (1 - stats.t.cdf(np.abs(t_statistic), DF))

        results = pd.DataFrame({'Continuous Variable': [num_var.name], 'Binary Variable': [bin_var.name], 'N': [n1+n2], 'Mean Diff': [f'{mean_diff:.2f}'], 
                            'SE of Mean Diff': [f'{SE_mean_diff:.2f}'], 'DF': [DF], 't-statistic': [f'{t_statistic:.3f}'],
                            'p-value': [f'{p_value:.3f}'], 'test': ["Welch's t-test"]})
        print("The homogeneity of variance assumption was violated, so Welch's t-test was conducted")
        
    return results

8. **Import the GED data set we used for the class activity. Call the t_test() function and test it out on the GED data set we used in class. Let the numeric variable be 'income_log' and let the binary variable be 'ged'.**

In [12]:
# import data 
data = pd.read_csv("../../Data/3 ged_data.csv")  

output = t_test(data['income_log'], data['ged'])

print(output)

The homogeneity of variance assumption was not violated, so an independent samples t-test was conducted
  Continuous Variable Binary Variable     N Mean Diff SE of Mean Diff    DF  \
0          income_log             ged  5976      0.48            0.06  5974   

  t-statistic p-value                        test  
0       7.458   0.000  Independent samples t-test  
